In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from src.dataset import (
    load_record,
    load_annotations
)

from src.preprocessing import (
    bandpass_filter,
    extract_beat_segments
)

from src.visualization import (
    plot_ecg
)

In [ ]:
record = load_record(
    "100"
)

signal = record.p_signal[:, 0]

fs = record.fs

filtered = bandpass_filter(
    signal,
    fs
)

In [ ]:
duration = 10

n = int(
    duration * fs
)

time = (
    np.arange(n) / fs
)

plt.figure(figsize=(14, 6))

plt.plot(
    time,
    signal[:n],
    label="Raw"
)

plt.plot(
    time,
    filtered[:n],
    label="Filtered"
)

plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude (mV)")
plt.title(
    "Raw vs Filtered ECG"
)

plt.legend()

plt.grid(True, alpha=0.3)

plt.tight_layout()

plt.show()

In [ ]:
annotation = load_annotations(
    "100"
)

X, y, symbols = extract_beat_segments(
    signal=filtered,
    annotation_samples=annotation.sample,
    annotation_symbols=annotation.symbol,
    fs=fs,
    before_seconds=0.2,
    after_seconds=0.4,
    normalize=True
)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of extracted beats:", len(X))

In [ ]:
plt.figure(figsize=(12, 6))

for i in range(
    min(10, len(X))
):

    plt.plot(
        X[i],
        alpha=0.6
    )

plt.xlabel("Sample")
plt.ylabel("Normalized amplitude")

plt.title(
    "Example Extracted ECG Beats"
)

plt.grid(True, alpha=0.3)

plt.tight_layout()

plt.show()

In [ ]:
class_names = {
    0: "N",
    1: "S",
    2: "V",
    3: "F",
    4: "Q",
}

plt.figure(
    figsize=(12, 8)
)

for class_id, class_name in class_names.items():

    indices = np.where(
        y == class_id
    )[0]

    if len(indices) == 0:
        continue

    index = indices[0]

    plt.plot(
        X[index],
        label=class_name
    )

plt.xlabel("Sample")
plt.ylabel("Normalized amplitude")

plt.title(
    "Representative ECG Beat Classes"
)

plt.legend()

plt.grid(True, alpha=0.3)

plt.tight_layout()

plt.show()

In [ ]:
from src.dataset import get_record_names
from src.preprocessing import process_all_records

records = get_record_names(
    str(DATA_DIR)
)

X, y, record_ids, symbols = process_all_records(
    records,
    str(DATA_DIR)
)

print("X:", X.shape)
print("y:", y.shape)
print("record_ids:", record_ids.shape)
print("symbols:", symbols.shape)

In [ ]:
BEATS_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "beats"
)

BEATS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
np.save(
    BEATS_DIR / "X.npy",
    X
)

np.save(
    BEATS_DIR / "y.npy",
    y
)

np.save(
    BEATS_DIR / "record_ids.npy",
    record_ids
)

np.save(
    BEATS_DIR / "symbols.npy",
    symbols
)

In [ ]:
X = np.load(
    BEATS_DIR / "X.npy"
)

y = np.load(
    BEATS_DIR / "y.npy"
)

record_ids = np.load(
    BEATS_DIR / "record_ids.npy"
)

print(X.shape)
print(y.shape)
print(record_ids.shape)

In [ ]:
from src.splits import split_records

train_records, val_records, test_records = split_records(
    records,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=42
)

print(
    "Train:",
    len(train_records),
    train_records
)

print(
    "Validation:",
    len(val_records),
    val_records
)

print(
    "Test:",
    len(test_records),
    test_records
)